# 🧠 GENESIS v5.0 — IA Auto-Evolutiva

**Notebook privado** para ejecutar Genesis en Google Colab con GPU gratuita.

## Instrucciones:
1. **Runtime > Change runtime type > T4 GPU**
2. Ejecuta las celdas en orden (Shift+Enter)
3. En la celda 2, sube el archivo `genesis_code.zip`
4. La ultima celda abre la Web UI

> Este notebook es **privado** — solo vos podes verlo desde tu cuenta Google.

## 1️⃣ Instalar Ollama + Dependencias

In [ ]:
# ============================================================
# CELDA 1: Instalar Ollama y dependencias
# ============================================================
import subprocess, os, time

print("[1/4] Instalando Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("\n[2/4] Iniciando Ollama server...")
# Lanzar ollama serve en background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Esperar que arranque

print("[3/4] Descargando modelo llama3.1 (~4.7GB, solo la primera vez)...")
!ollama pull llama3.1

print("\n[4/4] Instalando dependencias Python...")
!pip install -q flask psutil

# Verificar GPU
print("\n" + "="*50)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama list
print("="*50)
print("\n\u2705 Todo listo. Continua con la celda 2.")

## 2️⃣ Subir proyecto Genesis

In [ ]:
# ============================================================
# CELDA 2: Subir genesis_code.zip
# ============================================================
# Genera el zip en tu PC ejecutando en CMD:
#   cd "C:\Users\Lexus\Desktop\Nueva carpeta\GENESIS"
#   python create_colab_zip.py
#
# Esto crea genesis_code.zip (~5MB) sin los modelos pesados.
# ============================================================

import os, zipfile
from google.colab import files

print("Selecciona genesis_code.zip desde tu PC...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"\nArchivo recibido: {zip_name} ({len(uploaded[zip_name]) / 1024 / 1024:.1f} MB)")

# Extraer
GENESIS_DIR = "/content/GENESIS"
os.makedirs(GENESIS_DIR, exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(GENESIS_DIR)

# Verificar estructura
files_found = []
for root, dirs, fnames in os.walk(GENESIS_DIR):
    for f in fnames:
        if f.endswith('.py'):
            files_found.append(os.path.relpath(os.path.join(root, f), GENESIS_DIR))

print(f"\n\u2705 {len(files_found)} archivos Python extraidos en {GENESIS_DIR}")
print(f"Archivos principales:")
for f in sorted(files_found):
    if '/' not in f or f.startswith('core/'):
        if '/' not in f:
            print(f"  \u2022 {f}")
print(f"  \u2022 core/ ({len([f for f in files_found if f.startswith('core/')])} modulos)")
print(f"  \u2022 tests/ ({len([f for f in files_found if f.startswith('tests/')])} tests)")

## 3️⃣ Configurar Genesis para Ollama

In [ ]:
# ============================================================
# CELDA 3: Configurar Genesis para usar Ollama
# ============================================================
import os

GENESIS_DIR = "/content/GENESIS"
config_path = os.path.join(GENESIS_DIR, "config.py")

# Leer config actual
with open(config_path, 'r', encoding='utf-8') as f:
    config = f.read()

# Cambiar provider de 'local' a 'ollama'
config = config.replace(
    'LLM_PROVIDER = os.getenv("GENESIS_PROVIDER", "local")',
    'LLM_PROVIDER = os.getenv("GENESIS_PROVIDER", "ollama")'
)

# Guardar
with open(config_path, 'w', encoding='utf-8') as f:
    f.write(config)

print("\u2705 config.py actualizado:")
print("  Provider: ollama")
print("  Modelo: llama3.1")
print("  URL: http://localhost:11434")

# Verificar que Ollama responde
import urllib.request, json
try:
    req = urllib.request.Request("http://localhost:11434/api/tags")
    with urllib.request.urlopen(req, timeout=5) as resp:
        data = json.loads(resp.read())
        models = [m['name'] for m in data.get('models', [])]
        print(f"  Modelos disponibles: {models}")
        print("\n\u2705 Ollama funcionando. Continua con la celda 4.")
except Exception as e:
    print(f"\n\u26a0\ufe0f Ollama no responde: {e}")
    print("Ejecuta la celda 1 de nuevo.")

## 4️⃣ Verificar modulos (tests)

In [ ]:
# ============================================================
# CELDA 4: Ejecutar tests para verificar que todo funciona
# ============================================================
import subprocess, os, sys

GENESIS_DIR = "/content/GENESIS"
os.chdir(GENESIS_DIR)
sys.path.insert(0, GENESIS_DIR)

test_files = sorted([
    f for f in os.listdir('tests')
    if f.startswith('test_v') and f.endswith('.py')
])

print(f"Ejecutando {len(test_files)} test suites...\n")

total_passed = 0
total_failed = 0

for tf in test_files:
    result = subprocess.run(
        [sys.executable, f'tests/{tf}'],
        capture_output=True, text=True, timeout=60
    )
    output = result.stdout + result.stderr
    # Buscar linea de resultados
    for line in output.splitlines():
        if 'passed' in line.lower() or 'pasaron' in line.lower():
            # Extraer numeros
            import re
            nums = re.findall(r'(\d+)', line)
            if nums:
                passed = int(nums[0])
                failed = int(nums[1]) if len(nums) > 1 else 0
                total_passed += passed
                status = "\u2705" if failed == 0 else "\u274c"
                print(f"  {status} {tf}: {passed} passed, {failed} failed")
                break

print(f"\n{'='*50}")
print(f"  TOTAL: {total_passed} passed, {total_failed} failed")
print(f"{'='*50}")
if total_failed == 0:
    print("\n\u2705 Todos los tests pasaron. Continua con la celda 5.")
else:
    print("\n\u26a0\ufe0f Hay tests fallando. Revisa los errores arriba.")

## 5️⃣ Lanzar Genesis Web UI

In [ ]:
# ============================================================
# CELDA 5: Lanzar Web UI con tunel de Colab
# ============================================================
import subprocess, threading, time, os, sys

GENESIS_DIR = "/content/GENESIS"
os.chdir(GENESIS_DIR)
sys.path.insert(0, GENESIS_DIR)

# Modificar web_ui.py para que use 0.0.0.0 (necesario en Colab)
web_ui_path = os.path.join(GENESIS_DIR, "web_ui.py")
with open(web_ui_path, 'r', encoding='utf-8') as f:
    web_code = f.read()

# Asegurar que Flask escuche en todas las interfaces
if '127.0.0.1' in web_code:
    web_code = web_code.replace("host='127.0.0.1'", "host='0.0.0.0'")
    web_code = web_code.replace('host="127.0.0.1"', 'host="0.0.0.0"')
    with open(web_ui_path, 'w', encoding='utf-8') as f:
        f.write(web_code)

print("Iniciando Genesis Web UI...")
print("(Ollama + Llama 3.1 en GPU T4)\n")

# Lanzar web_ui.py en thread separado
def run_server():
    subprocess.run([sys.executable, web_ui_path], cwd=GENESIS_DIR)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Esperar a que arranque
print("Esperando que el servidor inicie...")
for i in range(60):
    time.sleep(2)
    try:
        import urllib.request
        urllib.request.urlopen("http://localhost:5000", timeout=2)
        break
    except:
        if i % 5 == 0:
            print(f"  Cargando... ({i*2}s)")

# Crear tunel con colab output
from google.colab import output
output.serve_kernel_port_as_window(5000)

print("\n" + "="*50)
print("  \u2705 GENESIS v5.0 Web UI activa!")
print("  Haz click en la URL de arriba para abrir.")
print("  (Solo vos podes acceder con tu cuenta Google)")
print("="*50)
print("\nComandos para probar en el chat:")
print("  /status     — Ver estado de 91 modulos")
print("  /help       — Lista de comandos")
print("  Hola!       — Conversar con Genesis")
print("\n\u26a0\ufe0f No cierres esta celda — mantiene el servidor corriendo.")

## 6️⃣ Chat directo (alternativa sin Web UI)

Si preferis chatear directamente en el notebook sin la Web UI:

In [ ]:
# ============================================================
# CELDA 6 (OPCIONAL): Chat directo en el notebook
# ============================================================
import os, sys

GENESIS_DIR = "/content/GENESIS"
os.chdir(GENESIS_DIR)
sys.path.insert(0, GENESIS_DIR)

from genesis import Genesis
from config import GENESIS_VERSION

genesis = Genesis()
print(f"\n\u2705 Genesis v{GENESIS_VERSION} listo!")
print("Escribe tu mensaje abajo. Escribe 'salir' para terminar.\n")

while True:
    try:
        user_input = input("\n\u27a4 Tu: ")
        if user_input.strip().lower() in ('salir', 'exit', 'quit'):
            genesis.process_input('/guardar')
            print("\nGenesis guardado. Hasta la proxima evolucion.")
            break
        response = genesis.process_input(user_input)
        print(f"\n\ud83e\udde0 Genesis: {response}")
    except KeyboardInterrupt:
        print("\nSesion interrumpida.")
        break